<a href="https://colab.research.google.com/github/pranathiperii/WHRadjBMI-adipose-regulatory-analysis/blob/main/WHRadjBMI_adipose_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# WHRadjBMI Adipose Regulatory Pipeline

This notebook automates the integration of GWAS, eQTL, and epigenomic
data to prioritise regulatory disease loci in adipose tissue.

It was built as part of a series of mini-projects analysing
BMI-adjusted waist-hip ratio (WHRadjBMI) GWAS signals.

## What this pipeline does

- Module 1 — filters and curates raw GWAS Catalog data
- Module 2 — retrieves adipose eQTL data from GTEx for candidate genes
- Module 3 — annotates SNPs with epigenomic regulatory evidence from ENCODE SCREEN

## Data sources
- GWAS Catalog (ebi.ac.uk/gwas)
- GTEx v8 (gtexportal.org)
- ENCODE SCREEN (screen.wenglab.org)

## Module 1 — GWAS data filtering and curation

Takes a raw GWAS Catalog associations file and filters it down to
genome-wide significant variants with the columns needed for
downstream analysis.

Input — raw .tsv file downloaded from GWAS Catalog
Output — cleaned, filtered table of significant SNPs

In [14]:
import pandas as pd
from google.colab import files

# upload your GWAS catalog file
uploaded = files.upload()

# read it in with latin-1 encoding to handle special characters
filename = list(uploaded.keys())[0]
gwas_raw = pd.read_csv(filename, sep='\t', low_memory=False, encoding='latin-1')

print(f"Raw file loaded — {len(gwas_raw)} associations")
print(f"\nColumns available:")
for col in gwas_raw.columns:
    print(f"  {col}")

Saving gwas-association-downloaded_2026-03-18-EFO_0004343-withChildTraits.tsv to gwas-association-downloaded_2026-03-18-EFO_0004343-withChildTraits (1).tsv
Raw file loaded — 10277 associations

Columns available:
  DATE ADDED TO CATALOG
  PUBMEDID
  FIRST AUTHOR
  DATE
  JOURNAL
  LINK
  STUDY
  DISEASE/TRAIT
  INITIAL SAMPLE SIZE
  REPLICATION SAMPLE SIZE
  REGION
  CHR_ID
  CHR_POS
  REPORTED GENE(S)
  MAPPED_GENE
  UPSTREAM_GENE_ID
  DOWNSTREAM_GENE_ID
  SNP_GENE_IDS
  UPSTREAM_GENE_DISTANCE
  DOWNSTREAM_GENE_DISTANCE
  STRONGEST SNP-RISK ALLELE
  SNPS
  MERGED
  SNP_ID_CURRENT
  CONTEXT
  INTERGENIC
  RISK ALLELE FREQUENCY
  P-VALUE
  PVALUE_MLOG
  P-VALUE (TEXT)
  OR or BETA
  95% CI (TEXT)
  PLATFORM [SNPS PASSING QC]
  CNV
  MAPPED_TRAIT
  MAPPED_TRAIT_URI
  STUDY ACCESSION
  GENOTYPING TECHNOLOGY


In [15]:
# Module 1 — filter and curate GWAS associations

# keep only the columns we need
columns_to_keep = [
    'REGION',
    'CHR_ID',
    'CHR_POS',
    'REPORTED GENE(S)',
    'MAPPED_GENE',
    'STRONGEST SNP-RISK ALLELE',
    'SNPS',
    'CONTEXT',
    'INTERGENIC',
    'RISK ALLELE FREQUENCY',
    'P-VALUE',
    'OR or BETA',
    'MAPPED_TRAIT'
]

gwas_filtered = gwas_raw[columns_to_keep].copy()

# filter to genome-wide significance
gwas_filtered['P-VALUE'] = pd.to_numeric(gwas_filtered['P-VALUE'], errors='coerce')
gwas_filtered = gwas_filtered[gwas_filtered['P-VALUE'] < 5e-8]

# filter to WHRadjBMI specifically
gwas_filtered = gwas_filtered[
    gwas_filtered['MAPPED_TRAIT'].str.contains('waist-hip ratio', case=False, na=False)
]

# drop duplicates — keep most significant per SNP
gwas_filtered = gwas_filtered.sort_values('P-VALUE')
gwas_filtered = gwas_filtered.drop_duplicates(subset='SNPS', keep='first')

# reset index
gwas_filtered = gwas_filtered.reset_index(drop=True)

print(f"Associations after filtering — {len(gwas_filtered)}")
print(f"\nTop 10 most significant:")
print(gwas_filtered[['REGION', 'MAPPED_GENE', 'SNPS', 'P-VALUE']].head(10))

Associations after filtering — 4692

Top 10 most significant:
    REGION        MAPPED_GENE         SNPS        P-VALUE
0  6q22.33              RSPO3   rs72959041  2.000000e-293
1  6q22.33              RSPO3  rs577721086  4.000000e-286
2   6p21.1  VEGFA - LINC02537     rs998584  3.000000e-168
3   6p21.1  VEGFA - LINC02537    rs6905288  9.000000e-140
4   2q24.3             COBLL1   rs13389219  5.000000e-127
5  6q22.33              RSPO3    rs1936805  2.000000e-120
6  6q22.33    RPS4XP9 - RSPO3  rs148306315  3.000000e-117
7  16q12.2                FTO    rs9923544  1.000000e-113
8  16q12.2                FTO    rs9940128  6.000000e-112
9  16q12.2                FTO   rs11075985  7.000000e-112


In [16]:
# tighten trait filter to BMI-adjusted specifically
gwas_adjbmi = gwas_filtered[
    gwas_filtered['MAPPED_TRAIT'].str.contains('body mass index', case=False, na=False) |
    gwas_filtered['MAPPED_TRAIT'].str.contains('BMI', case=False, na=False)
]

# if that removes too much, use the full filtered set instead
# check both
print(f"BMI-adjusted only — {len(gwas_adjbmi)} associations")
print(f"All WHR associations — {len(gwas_filtered)} associations")

print(f"\nUnique traits in filtered set:")
print(gwas_filtered['MAPPED_TRAIT'].value_counts())

BMI-adjusted only — 3925 associations
All WHR associations — 4692 associations

Unique traits in filtered set:
MAPPED_TRAIT
BMI-adjusted waist-hip ratio                                                    3750
waist-hip ratio                                                                  686
BMI-adjusted waist-hip ratio, colorectal cancer                                   45
BMI-adjusted waist-hip ratio, breast cancer                                       45
BMI-adjusted waist-hip ratio, physical activity measurement                       31
breast cancer, waist-hip ratio                                                    30
colorectal cancer, waist-hip ratio                                                27
BMI-adjusted waist-hip ratio, sex interaction measurement, age at assessment      16
BMI-adjusted waist-hip ratio, endometrial cancer                                   7
smoking behavior, BMI-adjusted waist-hip ratio                                     7
endometrial cancer, waist-

In [17]:
# keep only pure WHRadjBMI — no combined traits
gwas_final = gwas_filtered[
    gwas_filtered['MAPPED_TRAIT'] == 'BMI-adjusted waist-hip ratio'
].copy()

# reset index
gwas_final = gwas_final.reset_index(drop=True)

print(f"Final curated dataset — {len(gwas_final)} associations")
print(f"\nTop 15 most significant loci:")
print(gwas_final[['REGION', 'MAPPED_GENE', 'SNPS', 'P-VALUE', 'OR or BETA']].head(15))

# save to csv
gwas_final.to_csv('WHRadjBMI_curated_GWAS.csv', index=False)
print(f"\nSaved as WHRadjBMI_curated_GWAS.csv")

Final curated dataset — 3750 associations

Top 15 most significant loci:
      REGION        MAPPED_GENE         SNPS        P-VALUE  OR or BETA
0    6q22.33              RSPO3   rs72959041  2.000000e-293    0.162400
1    6q22.33              RSPO3  rs577721086  4.000000e-286         NaN
2     6p21.1  VEGFA - LINC02537     rs998584  3.000000e-168    0.050000
3     6p21.1  VEGFA - LINC02537    rs6905288  9.000000e-140    0.044400
4     2q24.3             COBLL1   rs13389219  5.000000e-127    0.068862
5    6q22.33              RSPO3    rs1936805  2.000000e-120    0.041000
6    6q22.33    RPS4XP9 - RSPO3  rs148306315  3.000000e-117    0.187336
7   12q24.31      RFLNA, ZNF664    rs7978610  4.000000e-101    0.063803
8   12q24.31              RFLNA     rs863750  4.000000e-101    0.037300
9    6q22.33    RPS4XP9 - RSPO3   rs10872311   4.000000e-99         NaN
10  12q24.31      RFLNA, ZNF664   rs11057413   2.000000e-97    0.062611
11  12q24.31      ZNF664, RFLNA     rs952632   5.000000e-97    

### Module 1 complete

The raw GWAS Catalog file has been filtered from 10,277 associations
down to 3,750 genome-wide significant WHRadjBMI associations.
Duplicate SNPs were removed keeping the most significant per locus.
Output saved as WHRadjBMI_curated_GWAS.csv.

## Module 2 — GTEx eQTL lookup

For each candidate gene, this module queries the GTEx API to retrieve
the top eQTL in adipose subcutaneous and adipose visceral tissue
automatically.

Instead of clicking through the GTEx portal manually for each gene,
the API returns the same data programmatically.

Input — list of candidate genes
Output — table of top eQTL SNP, p-value, effect size, and direction
per gene per tissue

In [18]:
import requests
import time

# your six candidate genes from mini-projects 1 and 2
candidate_genes = ['RSPO3', 'VEGFA', 'COBLL1', 'RFLNA', 'ZNF664', 'CCDC92']

# GTEx API endpoint for eQTL data
# we query each gene in adipose subcutaneous and visceral
tissues = [
    'Adipose_Subcutaneous',
    'Adipose_Visceral_Omentum'
]

def get_gtex_eqtl(gene_name, tissue):
    url = "https://gtexportal.org/api/v2/association/singleTissueEqtl"
    params = {
        'gencodeId': None,
        'tissueSiteDetailId': tissue,
        'datasetId': 'gtex_v8'
    }

    # first get the gencode ID for the gene
    gene_url = "https://gtexportal.org/api/v2/reference/gene"
    gene_params = {
        'geneSymbol': gene_name,
        'datasetId': 'gtex_v8'
    }

    gene_response = requests.get(gene_url, params=gene_params)

    if gene_response.status_code != 200:
        return None

    gene_data = gene_response.json()

    if not gene_data.get('data'):
        return None

    gencode_id = gene_data['data'][0]['gencodeId']

    # now get eQTL data
    params['gencodeId'] = gencode_id
    eqtl_response = requests.get(url, params=params)

    if eqtl_response.status_code != 200:
        return None

    eqtl_data = eqtl_response.json()

    if not eqtl_data.get('data'):
        return None

    # return the top hit (already sorted by p-value)
    top = eqtl_data['data'][0]
    return {
        'gene': gene_name,
        'tissue': tissue,
        'top_eqtl_snp': top.get('snpId'),
        'position': top.get('pos'),
        'pvalue': top.get('pval'),
        'nes': top.get('nes'),
        'ref_allele': top.get('ref'),
        'alt_allele': top.get('alt')
    }

# run for all genes and tissues
results = []

for gene in candidate_genes:
    for tissue in tissues:
        print(f"Querying {gene} in {tissue}...")
        result = get_gtex_eqtl(gene, tissue)
        if result:
            results.append(result)
        else:
            results.append({
                'gene': gene,
                'tissue': tissue,
                'top_eqtl_snp': 'not found',
                'position': None,
                'pvalue': None,
                'nes': None,
                'ref_allele': None,
                'alt_allele': None
            })
        time.sleep(0.5)  # be polite to the API

# convert to dataframe
eqtl_results = pd.DataFrame(results)
print("\nGTEx eQTL results:")
print(eqtl_results.to_string())

Querying RSPO3 in Adipose_Subcutaneous...
Querying RSPO3 in Adipose_Visceral_Omentum...
Querying VEGFA in Adipose_Subcutaneous...
Querying VEGFA in Adipose_Visceral_Omentum...
Querying COBLL1 in Adipose_Subcutaneous...
Querying COBLL1 in Adipose_Visceral_Omentum...
Querying RFLNA in Adipose_Subcutaneous...
Querying RFLNA in Adipose_Visceral_Omentum...
Querying ZNF664 in Adipose_Subcutaneous...
Querying ZNF664 in Adipose_Visceral_Omentum...
Querying CCDC92 in Adipose_Subcutaneous...
Querying CCDC92 in Adipose_Visceral_Omentum...

GTEx eQTL results:
      gene                    tissue top_eqtl_snp position pvalue   nes ref_allele alt_allele
0    RSPO3      Adipose_Subcutaneous    not found     None   None  None       None       None
1    RSPO3  Adipose_Visceral_Omentum    not found     None   None  None       None       None
2    VEGFA      Adipose_Subcutaneous    not found     None   None  None       None       None
3    VEGFA  Adipose_Visceral_Omentum    not found     None   None  Non

In [19]:
# debug — check what the gene lookup returns for RSPO3
import requests

gene_url = "https://gtexportal.org/api/v2/reference/gene"
gene_params = {
    'geneSymbol': 'RSPO3',
    'datasetId': 'gtex_v8'
}

response = requests.get(gene_url, params=gene_params)
print(f"Status code: {response.status_code}")
print(f"Response:")
print(response.json())

Status code: 422
Response:
{'detail': [{'type': 'missing', 'loc': ['query', 'geneId'], 'msg': 'Field required', 'input': None}]}


In [20]:
# try with geneId instead
gene_url = "https://gtexportal.org/api/v2/reference/gene"
gene_params = {
    'geneId': 'RSPO3',
    'datasetId': 'gtex_v8'
}

response = requests.get(gene_url, params=gene_params)
print(f"Status code: {response.status_code}")
print(response.json())

Status code: 200
{'data': [{'chromosome': 'chr6', 'dataSource': 'HAVANA', 'description': 'R-spondin 3 [Source:HGNC Symbol;Acc:HGNC:20866]', 'end': 127197765, 'entrezGeneId': 84870, 'gencodeId': 'ENSG00000146374.13', 'gencodeVersion': 'v26', 'geneStatus': '', 'geneSymbol': 'RSPO3', 'geneSymbolUpper': 'RSPO3', 'geneType': 'protein coding', 'genomeBuild': 'GRCh38/hg38', 'start': 127118604, 'strand': '+', 'tss': 127118604}], 'paging_info': {'numberOfPages': 1, 'page': 0, 'maxItemsPerPage': 250, 'totalNumberOfItems': 1}}


In [21]:
# test eQTL lookup with correct gencode ID
eqtl_url = "https://gtexportal.org/api/v2/association/singleTissueEqtl"
eqtl_params = {
    'gencodeId': 'ENSG00000146374.13',
    'tissueSiteDetailId': 'Adipose_Subcutaneous',
    'datasetId': 'gtex_v8'
}

response = requests.get(eqtl_url, params=eqtl_params)
print(f"Status code: {response.status_code}")
print(response.json())

Status code: 200
{'data': [{'snpId': 'rs141783576', 'pos': 127118752, 'snpIdUpper': 'RS141783576', 'variantId': 'chr6_127118752_G_C_b38', 'geneSymbol': 'RSPO3', 'pValue': 1.6819e-07, 'geneSymbolUpper': 'RSPO3', 'datasetId': 'gtex_v8', 'tissueSiteDetailId': 'Adipose_Subcutaneous', 'ontologyId': 'UBERON:0002190', 'chromosome': 'chr6', 'gencodeId': 'ENSG00000146374.13', 'nes': 0.463745}, {'snpId': 'rs577721086', 'pos': 127118902, 'snpIdUpper': 'RS577721086', 'variantId': 'chr6_127118902_T_C_b38', 'geneSymbol': 'RSPO3', 'pValue': 1.22264e-06, 'geneSymbolUpper': 'RSPO3', 'datasetId': 'gtex_v8', 'tissueSiteDetailId': 'Adipose_Subcutaneous', 'ontologyId': 'UBERON:0002190', 'chromosome': 'chr6', 'gencodeId': 'ENSG00000146374.13', 'nes': 0.496213}, {'snpId': 'rs9482770', 'pos': 127121947, 'snpIdUpper': 'RS9482770', 'variantId': 'chr6_127121947_T_C_b38', 'geneSymbol': 'RSPO3', 'pValue': 0.000128871, 'geneSymbolUpper': 'RSPO3', 'datasetId': 'gtex_v8', 'tissueSiteDetailId': 'Adipose_Subcutaneous',

In [22]:
# fixed function with correct API field names
def get_gtex_eqtl(gene_name, tissue):

    # step 1 — get gencode ID from gene symbol
    gene_url = "https://gtexportal.org/api/v2/reference/gene"
    gene_params = {
        'geneId': gene_name,
        'datasetId': 'gtex_v8'
    }

    gene_response = requests.get(gene_url, params=gene_params)

    if gene_response.status_code != 200:
        return None

    gene_data = gene_response.json()

    if not gene_data.get('data'):
        return None

    gencode_id = gene_data['data'][0]['gencodeId']

    # step 2 — get eQTL data
    eqtl_url = "https://gtexportal.org/api/v2/association/singleTissueEqtl"
    eqtl_params = {
        'gencodeId': gencode_id,
        'tissueSiteDetailId': tissue,
        'datasetId': 'gtex_v8'
    }

    eqtl_response = requests.get(eqtl_url, params=eqtl_params)

    if eqtl_response.status_code != 200:
        return None

    eqtl_data = eqtl_response.json()

    if not eqtl_data.get('data'):
        return None

    # sort by pValue and take the top hit
    hits = sorted(eqtl_data['data'], key=lambda x: x['pValue'])
    top = hits[0]

    return {
        'gene': gene_name,
        'tissue': tissue,
        'top_eqtl_snp': top.get('snpId'),
        'chromosome': top.get('chromosome'),
        'position': top.get('pos'),
        'variant_id': top.get('variantId'),
        'pvalue': top.get('pValue'),
        'nes': top.get('nes')
    }

# run for all genes and tissues
results = []

for gene in candidate_genes:
    for tissue in tissues:
        print(f"Querying {gene} in {tissue}...")
        result = get_gtex_eqtl(gene, tissue)
        if result:
            results.append(result)
        else:
            results.append({
                'gene': gene,
                'tissue': tissue,
                'top_eqtl_snp': 'not found',
                'chromosome': None,
                'position': None,
                'variant_id': None,
                'pvalue': None,
                'nes': None
            })
        time.sleep(0.5)

# convert to dataframe
eqtl_results = pd.DataFrame(results)

print("\nGTEx eQTL results:")
print(eqtl_results.to_string())

# save
eqtl_results.to_csv('WHRadjBMI_eQTL_results.csv', index=False)
print("\nSaved as WHRadjBMI_eQTL_results.csv")

Querying RSPO3 in Adipose_Subcutaneous...
Querying RSPO3 in Adipose_Visceral_Omentum...
Querying VEGFA in Adipose_Subcutaneous...
Querying VEGFA in Adipose_Visceral_Omentum...
Querying COBLL1 in Adipose_Subcutaneous...
Querying COBLL1 in Adipose_Visceral_Omentum...
Querying RFLNA in Adipose_Subcutaneous...
Querying RFLNA in Adipose_Visceral_Omentum...
Querying ZNF664 in Adipose_Subcutaneous...
Querying ZNF664 in Adipose_Visceral_Omentum...
Querying CCDC92 in Adipose_Subcutaneous...
Querying CCDC92 in Adipose_Visceral_Omentum...

GTEx eQTL results:
      gene                    tissue top_eqtl_snp chromosome     position               variant_id        pvalue       nes
0    RSPO3      Adipose_Subcutaneous  rs141783576       chr6  127118752.0   chr6_127118752_G_C_b38  1.681900e-07  0.463745
1    RSPO3  Adipose_Visceral_Omentum    not found       None          NaN                     None           NaN       NaN
2    VEGFA      Adipose_Subcutaneous  rs567694210       chr6   44340932.0    

In [23]:
# compare eQTL SNPs to GWAS lead SNPs
# your GWAS lead SNPs from mini-project 1
gwas_leads = {
    'RSPO3': {'snp': 'rs72959041', 'position': 127133748, 'chr': 'chr6'},
    'VEGFA': {'snp': 'rs6905288', 'position': 43791136, 'chr': 'chr6'},
    'COBLL1': {'snp': 'rs13389219', 'position': 164672366, 'chr': 'chr2'},
    'RFLNA': {'snp': 'rs863750', 'position': 124020897, 'chr': 'chr12'},
    'ZNF664': {'snp': 'rs7978610', 'position': 123984025, 'chr': 'chr12'},
    'CCDC92': {'snp': 'rs7133378', 'position': 123924955, 'chr': 'chr12'}
}

def classify_colocalisation(gene, eqtl_snp, eqtl_pos, gwas_leads):
    if gene not in gwas_leads:
        return 'unknown'

    if eqtl_snp == 'not found' or eqtl_pos is None:
        return 'no eQTL detected'

    gwas_snp = gwas_leads[gene]['snp']
    gwas_pos = gwas_leads[gene]['position']

    # same SNP
    if eqtl_snp == gwas_snp:
        return 'strong — same SNP'

    # calculate distance
    distance = abs(eqtl_pos - gwas_pos)

    if distance < 50000:
        return f'possible — {int(distance/1000)}kb apart'
    elif distance < 200000:
        return f'weak — {int(distance/1000)}kb apart'
    else:
        return f'unlikely — {int(distance/1000)}kb apart'

# apply classification
eqtl_results['gwas_lead_snp'] = eqtl_results['gene'].map(
    lambda g: gwas_leads.get(g, {}).get('snp', 'unknown')
)
eqtl_results['gwas_position'] = eqtl_results['gene'].map(
    lambda g: gwas_leads.get(g, {}).get('position', None)
)
eqtl_results['distance_from_gwas_kb'] = abs(
    eqtl_results['position'] - eqtl_results['gwas_position']
) / 1000

eqtl_results['colocalisation'] = eqtl_results.apply(
    lambda row: classify_colocalisation(
        row['gene'], row['top_eqtl_snp'], row['position'], gwas_leads
    ), axis=1
)

# print clean summary
summary = eqtl_results[[
    'gene', 'tissue', 'top_eqtl_snp', 'gwas_lead_snp',
    'pvalue', 'nes', 'distance_from_gwas_kb', 'colocalisation'
]]

print("Colocalisation summary:")
print(summary.to_string())

# save final output
summary.to_csv('WHRadjBMI_colocalisation_summary.csv', index=False)
print("\nSaved as WHRadjBMI_colocalisation_summary.csv")

Colocalisation summary:
      gene                    tissue top_eqtl_snp gwas_lead_snp        pvalue       nes  distance_from_gwas_kb          colocalisation
0    RSPO3      Adipose_Subcutaneous  rs141783576    rs72959041  1.681900e-07  0.463745                 14.996   possible — 14kb apart
1    RSPO3  Adipose_Visceral_Omentum    not found    rs72959041           NaN       NaN                    NaN        no eQTL detected
2    VEGFA      Adipose_Subcutaneous  rs567694210     rs6905288  1.004280e-05 -0.528589                549.796  unlikely — 549kb apart
3    VEGFA  Adipose_Visceral_Omentum   rs11571999     rs6905288  1.353710e-05  0.398557                630.314  unlikely — 630kb apart
4   COBLL1      Adipose_Subcutaneous   rs67970359    rs13389219  9.296230e-08 -0.201988                157.787      weak — 157kb apart
5   COBLL1  Adipose_Visceral_Omentum    rs3769885    rs13389219  4.180080e-07  0.130057                 63.514       weak — 63kb apart
6    RFLNA      Adipose_Subcuta

In [24]:
def get_gtex_eqtl_with_gwas_check(gene_name, tissue, gwas_snp):

    # get gencode ID
    gene_url = "https://gtexportal.org/api/v2/reference/gene"
    gene_response = requests.get(gene_url, params={
        'geneId': gene_name, 'datasetId': 'gtex_v8'
    })

    if gene_response.status_code != 200:
        return None

    gene_data = gene_response.json()
    if not gene_data.get('data'):
        return None

    gencode_id = gene_data['data'][0]['gencodeId']

    # get all eQTLs
    eqtl_url = "https://gtexportal.org/api/v2/association/singleTissueEqtl"
    eqtl_response = requests.get(eqtl_url, params={
        'gencodeId': gencode_id,
        'tissueSiteDetailId': tissue,
        'datasetId': 'gtex_v8'
    })

    if eqtl_response.status_code != 200:
        return None

    eqtl_data = eqtl_response.json()
    if not eqtl_data.get('data'):
        return None

    hits = eqtl_data['data']

    # check if GWAS lead SNP is anywhere in the eQTL results
    gwas_hit = None
    for hit in hits:
        if hit.get('snpId') == gwas_snp:
            gwas_hit = hit
            break

    # also get the top eQTL by p-value
    top_hit = sorted(hits, key=lambda x: x['pValue'])[0]

    # classify
    if gwas_hit:
        verdict = 'strong — same SNP in eQTL results'
        reporting_snp = gwas_hit['snpId']
        reporting_pvalue = gwas_hit['pValue']
        reporting_nes = gwas_hit['nes']
        reporting_pos = gwas_hit['pos']
    else:
        reporting_snp = top_hit['snpId']
        reporting_pvalue = top_hit['pValue']
        reporting_nes = top_hit['nes']
        reporting_pos = top_hit['pos']

        distance = abs(reporting_pos - gwas_leads[gene_name]['position'])
        if distance < 50000:
            verdict = f'possible — {int(distance/1000)}kb apart'
        elif distance < 200000:
            verdict = f'weak — {int(distance/1000)}kb apart'
        else:
            verdict = f'unlikely — {int(distance/1000)}kb apart'

    return {
        'gene': gene_name,
        'tissue': tissue,
        'gwas_lead_snp': gwas_snp,
        'gwas_snp_in_eqtl': gwas_hit is not None,
        'top_eqtl_snp': reporting_snp,
        'position': reporting_pos,
        'pvalue': reporting_pvalue,
        'nes': reporting_nes,
        'verdict': verdict
    }

# run updated version
results_v2 = []

for gene in candidate_genes:
    gwas_snp = gwas_leads[gene]['snp']
    for tissue in tissues:
        print(f"Querying {gene} in {tissue}...")
        result = get_gtex_eqtl_with_gwas_check(gene, tissue, gwas_snp)
        if result:
            results_v2.append(result)
        else:
            results_v2.append({
                'gene': gene,
                'tissue': tissue,
                'gwas_lead_snp': gwas_snp,
                'gwas_snp_in_eqtl': False,
                'top_eqtl_snp': 'not found',
                'position': None,
                'pvalue': None,
                'nes': None,
                'verdict': 'no eQTL detected'
            })
        time.sleep(0.5)

results_v2_df = pd.DataFrame(results_v2)

print("\nUpdated colocalisation summary:")
print(results_v2_df[[
    'gene', 'tissue', 'gwas_lead_snp',
    'gwas_snp_in_eqtl', 'top_eqtl_snp',
    'pvalue', 'nes', 'verdict'
]].to_string())

results_v2_df.to_csv('WHRadjBMI_colocalisation_final.csv', index=False)
print("\nSaved as WHRadjBMI_colocalisation_final.csv")

Querying RSPO3 in Adipose_Subcutaneous...
Querying RSPO3 in Adipose_Visceral_Omentum...
Querying VEGFA in Adipose_Subcutaneous...
Querying VEGFA in Adipose_Visceral_Omentum...
Querying COBLL1 in Adipose_Subcutaneous...
Querying COBLL1 in Adipose_Visceral_Omentum...
Querying RFLNA in Adipose_Subcutaneous...
Querying RFLNA in Adipose_Visceral_Omentum...
Querying ZNF664 in Adipose_Subcutaneous...
Querying ZNF664 in Adipose_Visceral_Omentum...
Querying CCDC92 in Adipose_Subcutaneous...
Querying CCDC92 in Adipose_Visceral_Omentum...

Updated colocalisation summary:
      gene                    tissue gwas_lead_snp  gwas_snp_in_eqtl top_eqtl_snp        pvalue       nes                            verdict
0    RSPO3      Adipose_Subcutaneous    rs72959041              True   rs72959041  6.185440e-07  0.509742  strong — same SNP in eQTL results
1    RSPO3  Adipose_Visceral_Omentum    rs72959041             False    not found           NaN       NaN                   no eQTL detected
2    VEGFA

### Module 2 complete

GTEx API was queried for all six candidate genes across adipose
subcutaneous and visceral omentum tissues. For each gene the pipeline
checks whether the GWAS lead SNP appears anywhere in the eQTL results,
not just whether it is the single top hit. This gives a more accurate
colocalisation verdict than distance-based comparison alone.

Output saved as WHRadjBMI_colocalisation_final.csv

Key findings:
- RSPO3 and CCDC92 show strong colocalisation — same SNP in both GWAS and adipose eQTL
- VEGFA signals are distant and unlikely to represent the same variant
- COBLL1 shows weak nearby signals in both depots
- RFLNA has an exceptionally strong independent visceral eQTL at a separate variant
- ZNF664 shows no direct overlap in either depot

## Module 3 — Epigenomic annotation using ENCODE SCREEN

For each GWAS lead SNP, this module queries the ENCODE SCREEN API to
check whether the SNP position overlaps a candidate cis-regulatory
element (cCRE) and what the chromatin evidence looks like.

This answers the question — is this SNP sitting in regulatory DNA or
just inactive sequence?

For each SNP we retrieve:
- Whether it overlaps a cCRE
- What type of cCRE (enhancer-like or promoter-like)
- The H3K27ac signal strength (active enhancer mark)
- The chromatin accessibility signal (open vs closed DNA)

Input — GWAS lead SNPs from Module 1
Output — regulatory classification for each SNP

In [25]:
import pandas as pd

# SCREEN epigenomic annotations
# retrieved manually via screen.wenglab.org for each GWAS lead SNP
# hardcoded here for reproducibility

screen_annotations = [
    {
        'gene': 'RSPO3',
        'snp': 'rs72959041',
        'chr': 'chr6',
        'position': 127133748,
        'ccre_overlap': True,
        'ccre_type': 'distal enhancer-like (dELS)',
        'h3k27ac': 'strong',
        'chromatin': 'open',
        'h3k4me1': 'present',
        'h3k4me3': 'absent',
        'regulatory_type': 'enhancer',
        'classification': 'strong',
        'notes': 'SNP overlaps cCRE with open chromatin and strong H3K27ac — classic active enhancer signature'
    },
    {
        'gene': 'VEGFA',
        'snp': 'rs6905288',
        'chr': 'chr6',
        'position': 43791136,
        'ccre_overlap': True,
        'ccre_type': 'enhancer-like',
        'h3k27ac': 'weak',
        'chromatin': 'open',
        'h3k4me1': 'present',
        'h3k4me3': 'absent',
        'regulatory_type': 'enhancer',
        'classification': 'weak/moderate',
        'notes': 'Open chromatin without strong H3K27ac — possible context-specific regulation'
    },
    {
        'gene': 'COBLL1',
        'snp': 'rs13389219',
        'chr': 'chr2',
        'position': 164672366,
        'ccre_overlap': True,
        'ccre_type': 'distal enhancer-like (dELS)',
        'h3k27ac': 'strong',
        'chromatin': 'open',
        'h3k4me1': 'present',
        'h3k4me3': 'absent',
        'regulatory_type': 'enhancer',
        'classification': 'strong',
        'notes': 'Clear ATAC + H3K27ac + H3K4me1 at SNP position — active enhancer regulation'
    },
    {
        'gene': 'RFLNA',
        'snp': 'rs863750',
        'chr': 'chr12',
        'position': 124020897,
        'ccre_overlap': True,
        'ccre_type': 'mixed',
        'h3k27ac': 'moderate',
        'chromatin': 'open',
        'h3k4me1': 'present',
        'h3k4me3': 'present',
        'regulatory_type': 'mixed enhancer/promoter',
        'classification': 'strong/mixed',
        'notes': 'Active chromatin marks present but signal not cleanly adipose-specific'
    },
    {
        'gene': 'ZNF664',
        'snp': 'rs7978610',
        'chr': 'chr12',
        'position': 123984025,
        'ccre_overlap': True,
        'ccre_type': 'distal enhancer-like (dELS)',
        'h3k27ac': 'strong',
        'chromatin': 'open',
        'h3k4me1': 'present',
        'h3k4me3': 'absent',
        'regulatory_type': 'enhancer',
        'classification': 'strong',
        'notes': 'SNP overlaps active enhancer-like region — absence of H3K4me3 confirms enhancer not promoter'
    },
    {
        'gene': 'CCDC92',
        'snp': 'rs7133378',
        'chr': 'chr12',
        'position': 123924955,
        'ccre_overlap': True,
        'ccre_type': 'distal enhancer-like (dELS)',
        'h3k27ac': 'strong',
        'chromatin': 'open',
        'h3k4me1': 'present',
        'h3k4me3': 'absent',
        'regulatory_type': 'enhancer',
        'classification': 'strong',
        'notes': 'Open chromatin and strong H3K27ac with ENCODE cCRE annotation — active enhancer'
    }
]

screen_df = pd.DataFrame(screen_annotations)

print("SCREEN epigenomic annotation results:")
print(screen_df[['gene', 'snp', 'ccre_overlap',
                  'regulatory_type', 'classification', 'notes']].to_string())

screen_df.to_csv('WHRadjBMI_SCREEN_annotation.csv', index=False)
print("\nSaved as WHRadjBMI_SCREEN_annotation.csv")

SCREEN epigenomic annotation results:
     gene         snp  ccre_overlap          regulatory_type classification                                                                                         notes
0   RSPO3  rs72959041          True                 enhancer         strong  SNP overlaps cCRE with open chromatin and strong H3K27ac — classic active enhancer signature
1   VEGFA   rs6905288          True                 enhancer  weak/moderate                  Open chromatin without strong H3K27ac — possible context-specific regulation
2  COBLL1  rs13389219          True                 enhancer         strong                   Clear ATAC + H3K27ac + H3K4me1 at SNP position — active enhancer regulation
3   RFLNA    rs863750          True  mixed enhancer/promoter   strong/mixed                        Active chromatin marks present but signal not cleanly adipose-specific
4  ZNF664   rs7978610          True                 enhancer         strong  SNP overlaps active enhancer-like r

In [26]:
# Final output — combine all three modules into one summary table
import pandas as pd

# reload saved outputs from previous modules
results_v2_df = pd.read_csv('WHRadjBMI_colocalisation_final.csv')
screen_df = pd.read_csv('WHRadjBMI_SCREEN_annotation.csv')

# gwas leads dictionary — needed for the merge
gwas_snps = {
    'RSPO3':  {'snp': 'rs72959041',  'chr': '6',  'position': 127133748},
    'VEGFA':  {'snp': 'rs6905288',   'chr': '6',  'position': 43791136},
    'COBLL1': {'snp': 'rs13389219',  'chr': '2',  'position': 164672366},
    'RFLNA':  {'snp': 'rs863750',    'chr': '12', 'position': 124020897},
    'ZNF664': {'snp': 'rs7978610',   'chr': '12', 'position': 123984025},
    'CCDC92': {'snp': 'rs7133378',   'chr': '12', 'position': 123924955}}
# start with GWAS leads
gwas_leads_df = pd.DataFrame([
    {'gene': gene, 'gwas_snp': info['snp'],
     'chr': f"chr{info['chr']}", 'position': info['position']}
    for gene, info in gwas_snps.items()
])

# merge with eQTL results — take subcutaneous first, visceral if no subq signal
eqtl_subq = results_v2_df[
    results_v2_df['tissue'] == 'Adipose_Subcutaneous'
][['gene', 'top_eqtl_snp', 'pvalue', 'nes', 'verdict']].copy()
eqtl_subq.columns = ['gene', 'eqtl_snp_subq', 'eqtl_pvalue_subq',
                      'eqtl_nes_subq', 'colocalisation_subq']

eqtl_visc = results_v2_df[
    results_v2_df['tissue'] == 'Adipose_Visceral_Omentum'
][['gene', 'top_eqtl_snp', 'pvalue', 'nes', 'verdict']].copy()
eqtl_visc.columns = ['gene', 'eqtl_snp_visc', 'eqtl_pvalue_visc',
                      'eqtl_nes_visc', 'colocalisation_visc']

# merge with SCREEN results
screen_summary = screen_df[['gene', 'regulatory_type',
                             'classification', 'notes']].copy()
screen_summary.columns = ['gene', 'regulatory_type',
                           'epigenomic_classification', 'epigenomic_notes']

# combine everything
final = gwas_leads_df.merge(eqtl_subq, on='gene')
final = final.merge(eqtl_visc, on='gene')
final = final.merge(screen_summary, on='gene')

# add overall verdict
def overall_verdict(row):
    if 'strong' in str(row['colocalisation_subq']) or \
       'strong' in str(row['colocalisation_visc']):
        return 'Strong — adipose colocalisation confirmed'
    elif 'possible' in str(row['colocalisation_subq']) or \
         'possible' in str(row['colocalisation_visc']):
        return 'Possible — nearby signal, formal colocalisation needed'
    elif 'weak' in str(row['colocalisation_subq']) or \
         'weak' in str(row['colocalisation_visc']):
        return 'Weak — signal present but distant'
    else:
        return 'Unlikely — no adipose colocalisation detected'

final['overall_verdict'] = final.apply(overall_verdict, axis=1)

print("FINAL INTEGRATED SUMMARY — WHRadjBMI adipose regulatory loci")
print("=" * 70)
for _, row in final.iterrows():
    print(f"\nGene: {row['gene']}")
    print(f"  GWAS SNP: {row['gwas_snp']} at {row['chr']}:{row['position']}")
    print(f"  Subcutaneous eQTL: {row['eqtl_snp_subq']} (p={row['eqtl_pvalue_subq']:.2e})"
          if row['eqtl_pvalue_subq'] else f"  Subcutaneous eQTL: not detected")
    print(f"  Visceral eQTL: {row['eqtl_snp_visc']} (p={row['eqtl_pvalue_visc']:.2e})"
          if row['eqtl_pvalue_visc'] else f"  Visceral eQTL: not detected")
    print(f"  Epigenomic support: {row['epigenomic_classification']}")
    print(f"  Overall verdict: {row['overall_verdict']}")

# save final output
final.to_csv('WHRadjBMI_integrated_summary.csv', index=False)
print("\n\nSaved as WHRadjBMI_integrated_summary.csv")
print("Pipeline complete.")

FINAL INTEGRATED SUMMARY — WHRadjBMI adipose regulatory loci

Gene: RSPO3
  GWAS SNP: rs72959041 at chr6:127133748
  Subcutaneous eQTL: rs72959041 (p=6.19e-07)
  Visceral eQTL: not found (p=nan)
  Epigenomic support: strong
  Overall verdict: Strong — adipose colocalisation confirmed

Gene: VEGFA
  GWAS SNP: rs6905288 at chr6:43791136
  Subcutaneous eQTL: rs567694210 (p=1.00e-05)
  Visceral eQTL: rs11571999 (p=1.35e-05)
  Epigenomic support: weak/moderate
  Overall verdict: Unlikely — no adipose colocalisation detected

Gene: COBLL1
  GWAS SNP: rs13389219 at chr2:164672366
  Subcutaneous eQTL: rs67970359 (p=9.30e-08)
  Visceral eQTL: rs3769885 (p=4.18e-07)
  Epigenomic support: strong
  Overall verdict: Weak — signal present but distant

Gene: RFLNA
  GWAS SNP: rs863750 at chr12:124020897
  Subcutaneous eQTL: rs2280534 (p=4.01e-05)
  Visceral eQTL: rs11057532 (p=1.01e-17)
  Epigenomic support: strong/mixed
  Overall verdict: Unlikely — no adipose colocalisation detected

Gene: ZNF664
 

## Pipeline complete

This notebook integrates three layers of genomic evidence for
WHRadjBMI GWAS candidate genes:

- Module 1 — GWAS Catalog filtering — 10,277 raw associations
  filtered to 3,750 genome-wide significant WHRadjBMI loci
- Module 2 — GTEx API eQTL lookup — automated colocalisation
  classification for adipose subcutaneous and visceral tissue
- Module 3 — ENCODE SCREEN epigenomic annotation — regulatory
  context for each GWAS lead SNP

Final output saved as WHRadjBMI_integrated_summary.csv

This pipeline can be applied to any GWAS trait by changing the
candidate gene list and GWAS lead SNPs in Modules 2 and 3.